In [1]:
import sys

print(sys.executable)

C:\Users\Varvara\Desktop\Repository\OMGTU\3\Methods\РГР\.venv\Scripts\python.exe


In [2]:
from sqlalchemy import (
    create_engine, Column, Integer, String, Numeric, Date, ForeignKey,
    Boolean, CheckConstraint, text
)
from sqlalchemy.orm import declarative_base, relationship, sessionmaker
from sqlalchemy.exc import SQLAlchemyError
from datetime import date
import pandas as pd
from sqlalchemy import text

print("Библиотеки успешно импортированы")

Библиотеки успешно импортированы


In [3]:
DB_USER = "postgres"
DB_PASSWORD = "postgres"
DB_HOST = "localhost"
DB_PORT = "5432"
DB_NAME = "cargo_transport_db"

DATABASE_URL = f"postgresql+psycopg2://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"

try:
    engine = create_engine(DATABASE_URL, echo=False)
    with engine.connect() as connection:
        print("Подключение к PostgreSQL успешно выполнено")
except SQLAlchemyError as error:
    print("Ошибка подключения к базе данных:")
    print(error)

Подключение к PostgreSQL успешно выполнено


## Создание ORM-моделей базы данных

На данном этапе описывается структура реляционной базы данных средствами SQLAlchemy ORM. 
База данных содержит таблицы маршрутов, водителей, коэффициентов оплаты по стажу, перевозок и связи перевозок с водителями.

In [4]:
from sqlalchemy import UniqueConstraint

Base = declarative_base()


class Route(Base):
    __tablename__ = "routes"

    id = Column("route_id", Integer, primary_key=True)
    name = Column(String(120), nullable=False, unique=True)
    distance = Column(Numeric(10, 2), nullable=False)
    base_salary = Column(Numeric(12, 2), nullable=False)

    shipments = relationship("Shipment", back_populates="route")

    __table_args__ = (
        CheckConstraint("distance > 0", name="check_route_distance_positive"),
        CheckConstraint("base_salary > 0", name="check_route_base_salary_positive"),
    )

    def __repr__(self):
        return f"<Route(id={self.id}, name='{self.name}', distance={self.distance})>"


class ExperienceCoefficient(Base):
    __tablename__ = "experience_coefficients"

    id = Column("experience_level_id", Integer, primary_key=True)
    percentage_rate = Column(Numeric(5, 2), nullable=False)
    description = Column(String(120), nullable=False)

    drivers = relationship("Driver", back_populates="experience_level")

    __table_args__ = (
        CheckConstraint("percentage_rate >= 0", name="check_percentage_rate_non_negative"),
    )

    def __repr__(self):
        return f"<ExperienceCoefficient(id={self.id}, percentage_rate={self.percentage_rate})>"


class Driver(Base):
    __tablename__ = "drivers"

    id = Column("driver_id", Integer, primary_key=True)
    experience_level_id = Column(
        Integer,
        ForeignKey("experience_coefficients.experience_level_id"),
        nullable=False
    )
    full_name = Column(String(255), nullable=False)

    experience_level = relationship("ExperienceCoefficient", back_populates="drivers")
    shipment_links = relationship("ShipmentDriver", back_populates="driver")

    def __repr__(self):
        return f"<Driver(id={self.id}, full_name='{self.full_name}')>"


class Shipment(Base):
    __tablename__ = "shipments"

    id = Column("shipment_id", Integer, primary_key=True)
    route_id = Column(Integer, ForeignKey("routes.route_id"), nullable=False)
    departure_date = Column(Date, nullable=False)
    arrival_date = Column(Date, nullable=False)

    route = relationship("Route", back_populates="shipments")
    driver_links = relationship("ShipmentDriver", back_populates="shipment", cascade="all, delete-orphan")

    __table_args__ = (
        CheckConstraint("arrival_date >= departure_date", name="check_arrival_after_departure"),
    )

    def __repr__(self):
        return f"<Shipment(id={self.id}, route_id={self.route_id})>"


class ShipmentDriver(Base):
    __tablename__ = "shipment_drivers"

    shipment_id = Column(Integer, ForeignKey("shipments.shipment_id"), primary_key=True)
    driver_id = Column(Integer, ForeignKey("drivers.driver_id"), primary_key=True)
    bonus = Column(Numeric(12, 2), nullable=False, default=0)

    shipment = relationship("Shipment", back_populates="driver_links")
    driver = relationship("Driver", back_populates="shipment_links")

    __table_args__ = (
        CheckConstraint("bonus >= 0", name="check_bonus_non_negative"),
    )

    def __repr__(self):
        return f"<ShipmentDriver(shipment_id={self.shipment_id}, driver_id={self.driver_id}, bonus={self.bonus})>"


print("ORM-модели успешно описаны")

ORM-модели успешно описаны


In [5]:
try:
    with engine.begin() as connection:
        connection.execute(text("DROP VIEW IF EXISTS shipment_payment_view CASCADE;"))

        connection.execute(text("""
            DO $$
            BEGIN
                IF to_regclass('public.shipment_drivers') IS NOT NULL THEN
                    DROP TRIGGER IF EXISTS check_max_two_drivers_trigger ON shipment_drivers;
                END IF;
            END $$;
        """))

        connection.execute(text("DROP FUNCTION IF EXISTS check_max_two_drivers() CASCADE;"))
        connection.execute(text("DROP FUNCTION IF EXISTS calculate_driver_payment(NUMERIC, INTEGER, NUMERIC);"))
        connection.execute(text("DROP PROCEDURE IF EXISTS add_bonus_to_driver(INTEGER, INTEGER, NUMERIC);"))

    Base.metadata.drop_all(engine)
    Base.metadata.create_all(engine)

    print("Таблицы успешно созданы в базе данных")

except SQLAlchemyError as error:
    print("Ошибка при создании таблиц:")
    print(error)

Таблицы успешно созданы в базе данных


Проверка

In [6]:
with engine.connect() as connection:
    result = connection.execute(text("""
        SELECT table_name
        FROM information_schema.tables
        WHERE table_schema = 'public'
        ORDER BY table_name;
    """))

    for row in result:
        print(row[0])

drivers
experience_coefficients
routes
shipment_drivers
shipments


In [7]:
from sqlalchemy import text

with engine.begin() as connection:
    connection.execute(text("ALTER USER postgres WITH PASSWORD 'postgres';"))

print("Пароль пользователя postgres изменен на postgres")

Пароль пользователя postgres изменен на postgres


## Заполнение базы данных тестовыми данными

На данном этапе в таблицы добавляются тестовые данные: маршруты, водители, коэффициенты оплаты по стажу, перевозки и связи между перевозками и водителями.

In [8]:
SessionLocal = sessionmaker(bind=engine)

print("Фабрика сессий успешно создана")

Фабрика сессий успешно создана


In [9]:
from decimal import Decimal

try:
    session = SessionLocal()

    session.query(ShipmentDriver).delete()
    session.query(Shipment).delete()
    session.query(Driver).delete()
    session.query(ExperienceCoefficient).delete()
    session.query(Route).delete()
    session.commit()

    routes = [
        Route(name="Омск — Новосибирск", distance=Decimal("650.00"), base_salary=Decimal("15000.00")),
        Route(name="Омск — Тюмень", distance=Decimal("620.00"), base_salary=Decimal("14500.00")),
        Route(name="Омск — Екатеринбург", distance=Decimal("950.00"), base_salary=Decimal("22000.00")),
        Route(name="Новосибирск — Красноярск", distance=Decimal("790.00"), base_salary=Decimal("18500.00")),
        Route(name="Омск — Москва", distance=Decimal("2700.00"), base_salary=Decimal("55000.00")),
    ]

    experience_levels = [
        ExperienceCoefficient(percentage_rate=Decimal("0.00"), description="Начальный стаж"),
        ExperienceCoefficient(percentage_rate=Decimal("10.00"), description="Средний стаж"),
        ExperienceCoefficient(percentage_rate=Decimal("25.00"), description="Опытный водитель"),
        ExperienceCoefficient(percentage_rate=Decimal("40.00"), description="Высокий стаж"),
    ]

    session.add_all(routes)
    session.add_all(experience_levels)
    session.commit()

    drivers = [
        Driver(full_name="Иванов Иван Петрович", experience_level_id=experience_levels[0].id),
        Driver(full_name="Петров Алексей Сергеевич", experience_level_id=experience_levels[1].id),
        Driver(full_name="Сидоров Дмитрий Андреевич", experience_level_id=experience_levels[2].id),
        Driver(full_name="Кузнецов Олег Викторович", experience_level_id=experience_levels[3].id),
        Driver(full_name="Смирнов Николай Игоревич", experience_level_id=experience_levels[3].id),
    ]

    session.add_all(drivers)
    session.commit()

    shipments = [
        Shipment(route_id=routes[0].id, departure_date=date(2026, 3, 1), arrival_date=date(2026, 3, 2)),
        Shipment(route_id=routes[1].id, departure_date=date(2026, 3, 3), arrival_date=date(2026, 3, 4)),
        Shipment(route_id=routes[2].id, departure_date=date(2026, 3, 5), arrival_date=date(2026, 3, 7)),
        Shipment(route_id=routes[3].id, departure_date=date(2026, 3, 8), arrival_date=date(2026, 3, 10)),
        Shipment(route_id=routes[4].id, departure_date=date(2026, 3, 11), arrival_date=date(2026, 3, 15)),
    ]

    session.add_all(shipments)
    session.commit()

    shipment_drivers = [
        ShipmentDriver(shipment_id=shipments[0].id, driver_id=drivers[0].id, bonus=Decimal("1000.00")),

        ShipmentDriver(shipment_id=shipments[1].id, driver_id=drivers[1].id, bonus=Decimal("1500.00")),
        ShipmentDriver(shipment_id=shipments[1].id, driver_id=drivers[2].id, bonus=Decimal("1000.00")),

        ShipmentDriver(shipment_id=shipments[2].id, driver_id=drivers[2].id, bonus=Decimal("2000.00")),

        ShipmentDriver(shipment_id=shipments[3].id, driver_id=drivers[3].id, bonus=Decimal("2500.00")),
        ShipmentDriver(shipment_id=shipments[3].id, driver_id=drivers[4].id, bonus=Decimal("2000.00")),

        ShipmentDriver(shipment_id=shipments[4].id, driver_id=drivers[4].id, bonus=Decimal("5000.00")),
    ]

    session.add_all(shipment_drivers)
    session.commit()

    print("Тестовые данные успешно добавлены")

except SQLAlchemyError as error:
    session.rollback()
    print("Ошибка при добавлении данных:")
    print(error)

finally:
    session.close()

Тестовые данные успешно добавлены


In [10]:
with engine.connect() as connection:
    tables = ["routes", "experience_coefficients", "drivers", "shipments", "shipment_drivers"]

    for table in tables:
        result = connection.execute(text(f"SELECT COUNT(*) FROM {table};"))
        count = result.scalar()
        print(f"{table}: {count} записей")

routes: 5 записей
experience_coefficients: 4 записей
drivers: 5 записей
shipments: 5 записей
shipment_drivers: 7 записей


In [11]:
def show_query(sql):
    with engine.connect() as connection:
        return pd.read_sql(text(sql), connection)

print("Функция show_query успешно создана")

Функция show_query успешно создана


Проверка

In [12]:
tables = {
    "routes": "route_id",
    "experience_coefficients": "experience_level_id",
    "drivers": "driver_id",
    "shipments": "shipment_id",
    "shipment_drivers": "shipment_id, driver_id",
}

for table_name, order_columns in tables.items():
    print(f"\nТаблица: {table_name}")
    
    display(
        show_query(
            f"""
            SELECT *
            FROM {table_name}
            ORDER BY {order_columns};
            """
        )
    )


Таблица: routes


,route_id,name,distance,base_salary
0,1,Омск — Новосибирск,650.0,15000.0
1,2,Омск — Тюмень,620.0,14500.0
2,3,Омск — Екатеринбург,950.0,22000.0
3,4,Новосибирск — Красноярск,790.0,18500.0
4,5,Омск — Москва,2700.0,55000.0



Таблица: experience_coefficients


,experience_level_id,percentage_rate,description
0,1,0.0,Начальный стаж
1,2,10.0,Средний стаж
2,3,25.0,Опытный водитель
3,4,40.0,Высокий стаж



Таблица: drivers


,driver_id,experience_level_id,full_name
0,1,1,Иванов Иван Петрович
1,2,2,Петров Алексей Сергеевич
2,3,3,Сидоров Дмитрий Андреевич
3,4,4,Кузнецов Олег Викторович
4,5,4,Смирнов Николай Игоревич



Таблица: shipments


,shipment_id,route_id,departure_date,arrival_date
0,1,1,2026-03-01,2026-03-02
1,2,2,2026-03-03,2026-03-04
2,3,3,2026-03-05,2026-03-07
3,4,4,2026-03-08,2026-03-10
4,5,5,2026-03-11,2026-03-15



Таблица: shipment_drivers


,shipment_id,driver_id,bonus
0,1,1,1000.0
1,2,2,1500.0
2,2,3,1000.0
3,3,3,2000.0
4,4,4,2500.0
5,4,5,2000.0
6,5,5,5000.0


## CRUD-операции с использованием SQLAlchemy ORM

В данном разделе выполняются основные операции с данными: добавление, чтение, изменение и удаление записей. 
Все операции выполняются через SQLAlchemy ORM.

CREATE — добавление водителя

In [13]:
try:
    session = SessionLocal()

    new_driver = Driver(
        full_name="Тестов Андрей Владимирович",
        experience_level_id=2
    )

    session.add(new_driver)
    session.commit()

    created_driver_id = new_driver.id

    print(f"Водитель успешно добавлен. ID нового водителя: {created_driver_id}")

except SQLAlchemyError as error:
    session.rollback()
    print("Ошибка при добавлении водителя:")
    print(error)

finally:
    session.close()

Водитель успешно добавлен. ID нового водителя: 6


READ — чтение водителей

In [14]:
try:
    session = SessionLocal()

    drivers_list = session.query(Driver).order_by(Driver.id).all()

    for driver in drivers_list:
        print(
            driver.id,
            driver.full_name,
            f"категория стажа: {driver.experience_level.description}",
            f"процентная ставка: {driver.experience_level.percentage_rate}%"
        )

except SQLAlchemyError as error:
    print("Ошибка при чтении данных:")
    print(error)

finally:
    session.close()

1 Иванов Иван Петрович категория стажа: Начальный стаж процентная ставка: 0.00%
2 Петров Алексей Сергеевич категория стажа: Средний стаж процентная ставка: 10.00%
3 Сидоров Дмитрий Андреевич категория стажа: Опытный водитель процентная ставка: 25.00%
4 Кузнецов Олег Викторович категория стажа: Высокий стаж процентная ставка: 40.00%
5 Смирнов Николай Игоревич категория стажа: Высокий стаж процентная ставка: 40.00%
6 Тестов Андрей Владимирович категория стажа: Средний стаж процентная ставка: 10.00%


UPDATE — изменение стажа водителя

In [15]:
try:
    session = SessionLocal()

    driver = session.query(Driver).filter(Driver.id == created_driver_id).first()

    if driver:
        driver.experience_level_id = 3
        session.commit()
        print(f"Категория стажа водителя {driver.full_name} изменена")
    else:
        print("Водитель не найден")

except SQLAlchemyError as error:
    session.rollback()
    print("Ошибка при изменении данных:")
    print(error)

finally:
    session.close()

Категория стажа водителя Тестов Андрей Владимирович изменена


Проверка после UPDATE

In [16]:
pd.read_sql(
    f"""
    SELECT 
        d.driver_id,
        d.full_name,
        ec.description,
        ec.percentage_rate
    FROM drivers d
    JOIN experience_coefficients ec ON d.experience_level_id = ec.experience_level_id
    WHERE d.driver_id = {created_driver_id}
    """,
    engine
)

,driver_id,full_name,description,percentage_rate
0,6,Тестов Андрей Владимирович,Опытный водитель,25.0


DELETE — удаление тестового водителя

In [17]:
try:
    session = SessionLocal()

    driver = session.query(Driver).filter(Driver.id == created_driver_id).first()

    if driver:
        session.delete(driver)
        session.commit()
        print(f"Водитель с ID {created_driver_id} успешно удален")
    else:
        print("Водитель уже был удален или не найден")

except SQLAlchemyError as error:
    session.rollback()
    print("Ошибка при удалении данных:")
    print(error)

finally:
    session.close()

Водитель с ID 6 успешно удален


Проверка после DELETE

In [18]:
show_query(f"""
SELECT
    d.driver_id,
    d.full_name,
    ec.description AS experience_level,
    ec.percentage_rate
FROM drivers d
JOIN experience_coefficients ec
    ON d.experience_level_id = ec.experience_level_id
WHERE d.driver_id = {created_driver_id};
""")

,driver_id,full_name,experience_level,percentage_rate


## Сложные запросы на выборку данных

В данном разделе выполняются сложные SQL-запросы к базе данных. 
Запросы позволяют получить информацию о перевозках, водителях, маршрутах и рассчитать выплаты водителям с учетом маршрута, стажа и премии.

Вспомогательная функция для красивого вывода

In [19]:
def show_query(sql):
    with engine.connect() as connection:
        return pd.read_sql(text(sql), connection)

print("Функция для выполнения SQL-запросов создана")

Функция для выполнения SQL-запросов создана


Запрос 1 — список перевозок с маршрутами и водителями

In [20]:
sql = """
SELECT 
    s.shipment_id,
    r.name AS route_name,
    r.distance,
    s.departure_date,
    s.arrival_date,
    STRING_AGG(d.full_name, ', ') AS drivers
FROM shipments s
JOIN routes r ON s.route_id = r.route_id
JOIN shipment_drivers sd ON s.shipment_id = sd.shipment_id
JOIN drivers d ON sd.driver_id = d.driver_id
GROUP BY 
    s.shipment_id, r.name, r.distance, 
    s.departure_date, s.arrival_date
ORDER BY s.shipment_id;
"""

show_query(sql)

,shipment_id,route_name,distance,departure_date,arrival_date,drivers
0,1,Омск — Новосибирск,650.0,2026-03-01,2026-03-02,Иванов Иван Петрович
1,2,Омск — Тюмень,620.0,2026-03-03,2026-03-04,"Петров Алексей Сергеевич, Сидоров Дмитрий Андр..."
2,3,Омск — Екатеринбург,950.0,2026-03-05,2026-03-07,Сидоров Дмитрий Андреевич
3,4,Новосибирск — Красноярск,790.0,2026-03-08,2026-03-10,"Кузнецов Олег Викторович, Смирнов Николай Игор..."
4,5,Омск — Москва,2700.0,2026-03-11,2026-03-15,Смирнов Николай Игоревич


Запрос 2 — расчет выплаты каждому водителю

In [21]:
sql = """
SELECT
    s.shipment_id,
    r.name AS route_name,
    d.full_name AS driver_full_name,
    ec.description AS experience_level,
    ec.percentage_rate,
    r.base_salary,
    sd.bonus,
    ROUND(r.base_salary + r.base_salary * ec.percentage_rate / 100 + sd.bonus, 2) AS final_payment
FROM shipment_drivers sd
JOIN shipments s ON sd.shipment_id = s.shipment_id
JOIN routes r ON s.route_id = r.route_id
JOIN drivers d ON sd.driver_id = d.driver_id
JOIN experience_coefficients ec ON d.experience_level_id = ec.experience_level_id
ORDER BY s.shipment_id, driver_full_name;
"""

show_query(sql)

,shipment_id,route_name,driver_full_name,experience_level,percentage_rate,base_salary,bonus,final_payment
0,1,Омск — Новосибирск,Иванов Иван Петрович,Начальный стаж,0.0,15000.0,1000.0,16000.0
1,2,Омск — Тюмень,Петров Алексей Сергеевич,Средний стаж,10.0,14500.0,1500.0,17450.0
2,2,Омск — Тюмень,Сидоров Дмитрий Андреевич,Опытный водитель,25.0,14500.0,1000.0,19125.0
3,3,Омск — Екатеринбург,Сидоров Дмитрий Андреевич,Опытный водитель,25.0,22000.0,2000.0,29500.0
4,4,Новосибирск — Красноярск,Кузнецов Олег Викторович,Высокий стаж,40.0,18500.0,2500.0,28400.0
5,4,Новосибирск — Красноярск,Смирнов Николай Игоревич,Высокий стаж,40.0,18500.0,2000.0,27900.0
6,5,Омск — Москва,Смирнов Николай Игоревич,Высокий стаж,40.0,55000.0,5000.0,82000.0


Запрос 3 — общая стоимость выплат по каждой перевозке

In [22]:
sql = """
SELECT
    s.shipment_id,
    r.name AS route_name,
    COUNT(sd.driver_id) AS drivers_count,
    SUM(ROUND(r.base_salary + r.base_salary * ec.percentage_rate / 100 + sd.bonus, 2)) AS total_drivers_payment
FROM shipments s
JOIN routes r ON s.route_id = r.route_id
JOIN shipment_drivers sd ON s.shipment_id = sd.shipment_id
JOIN drivers d ON sd.driver_id = d.driver_id
JOIN experience_coefficients ec ON d.experience_level_id = ec.experience_level_id
GROUP BY s.shipment_id, r.name
ORDER BY total_drivers_payment DESC;
"""

show_query(sql)

,shipment_id,route_name,drivers_count,total_drivers_payment
0,5,Омск — Москва,1,82000.0
1,4,Новосибирск — Красноярск,2,56300.0
2,2,Омск — Тюмень,2,36575.0
3,3,Омск — Екатеринбург,1,29500.0
4,1,Омск — Новосибирск,1,16000.0


Запрос 4 — общий заработок каждого водителя

In [23]:
sql = """
SELECT
    d.driver_id,
    d.full_name AS driver_full_name,
    ec.description AS experience_level,
    ec.percentage_rate,
    COUNT(s.shipment_id) AS shipments_count,
    SUM(ROUND(r.base_salary + r.base_salary * ec.percentage_rate / 100 + sd.bonus, 2)) AS total_earned
FROM drivers d
JOIN experience_coefficients ec ON d.experience_level_id = ec.experience_level_id
JOIN shipment_drivers sd ON d.driver_id = sd.driver_id
JOIN shipments s ON sd.shipment_id = s.shipment_id
JOIN routes r ON s.route_id = r.route_id
GROUP BY d.driver_id, d.full_name, ec.description, ec.percentage_rate
ORDER BY total_earned DESC;
"""

show_query(sql)

,driver_id,driver_full_name,experience_level,percentage_rate,shipments_count,total_earned
0,5,Смирнов Николай Игоревич,Высокий стаж,40.0,2,109900.0
1,3,Сидоров Дмитрий Андреевич,Опытный водитель,25.0,2,48625.0
2,4,Кузнецов Олег Викторович,Высокий стаж,40.0,1,28400.0
3,2,Петров Алексей Сергеевич,Средний стаж,10.0,1,17450.0
4,1,Иванов Иван Петрович,Начальный стаж,0.0,1,16000.0


Запрос 5 — перевозки, где участвовали два водителя

In [24]:
sql = """
SELECT
    s.shipment_id,
    r.name AS route_name,
    COUNT(sd.driver_id) AS drivers_count,
    STRING_AGG(d.full_name, ', ') AS drivers
FROM shipments s
JOIN routes r ON s.route_id = r.route_id
JOIN shipment_drivers sd ON s.shipment_id = sd.shipment_id
JOIN drivers d ON sd.driver_id = d.driver_id
GROUP BY s.shipment_id, r.name
HAVING COUNT(sd.driver_id) = 2
ORDER BY s.shipment_id;
"""

show_query(sql)

,shipment_id,route_name,drivers_count,drivers
0,2,Омск — Тюмень,2,"Петров Алексей Сергеевич, Сидоров Дмитрий Андр..."
1,4,Новосибирск — Красноярск,2,"Кузнецов Олег Викторович, Смирнов Николай Игор..."


Запрос 6 — самые дорогие маршруты по выплатам

In [25]:
sql = """
SELECT
    r.route_id,
    r.name AS route_name,
    r.distance,
    r.base_salary,
    COUNT(s.shipment_id) AS shipments_count,
    SUM(ROUND(r.base_salary + r.base_salary * ec.percentage_rate / 100 + sd.bonus, 2)) AS total_payments
FROM routes r
JOIN shipments s ON r.route_id = s.route_id
JOIN shipment_drivers sd ON s.shipment_id = sd.shipment_id
JOIN drivers d ON sd.driver_id = d.driver_id
JOIN experience_coefficients ec ON d.experience_level_id = ec.experience_level_id
GROUP BY r.route_id, r.name, r.distance, r.base_salary
ORDER BY total_payments DESC;
"""

show_query(sql)

,route_id,route_name,distance,base_salary,shipments_count,total_payments
0,5,Омск — Москва,2700.0,55000.0,1,82000.0
1,4,Новосибирск — Красноярск,790.0,18500.0,2,56300.0
2,2,Омск — Тюмень,620.0,14500.0,2,36575.0
3,3,Омск — Екатеринбург,950.0,22000.0,1,29500.0
4,1,Омск — Новосибирск,650.0,15000.0,1,16000.0


## Создание представления для расчета выплат водителям

Представление используется для объединения данных о перевозках, маршрутах, водителях, стаже, премиях и итоговых выплатах. 
Оно позволяет обращаться к расчетным данным как к обычной таблице.

In [26]:
try:
    with engine.begin() as connection:
        connection.execute(text("""
            DROP VIEW IF EXISTS shipment_payment_view;
        """))

        connection.execute(text("""
            CREATE VIEW shipment_payment_view AS
            SELECT
                s.shipment_id,
                r.name AS route_name,
                r.distance,
                s.departure_date,
                s.arrival_date,
                d.driver_id,
                d.full_name AS driver_full_name,
                ec.description AS experience_level,
                ec.percentage_rate,
                r.base_salary,
                sd.bonus,
                ROUND(r.base_salary + r.base_salary * ec.percentage_rate / 100 + sd.bonus, 2) AS final_payment
            FROM shipment_drivers sd
            JOIN shipments s ON sd.shipment_id = s.shipment_id
            JOIN routes r ON s.route_id = r.route_id
            JOIN drivers d ON sd.driver_id = d.driver_id
            JOIN experience_coefficients ec ON d.experience_level_id = ec.experience_level_id;
        """))

    print("Представление shipment_payment_view успешно создано")

except SQLAlchemyError as error:
    print("Ошибка при создании представления:")
    print(error)

Представление shipment_payment_view успешно создано


Проверяем VIEW

In [27]:
show_query("""
SELECT *
FROM shipment_payment_view
ORDER BY shipment_id, driver_full_name;
""")

,shipment_id,route_name,distance,departure_date,arrival_date,driver_id,driver_full_name,experience_level,percentage_rate,base_salary,bonus,final_payment
0,1,Омск — Новосибирск,650.0,2026-03-01,2026-03-02,1,Иванов Иван Петрович,Начальный стаж,0.0,15000.0,1000.0,16000.0
1,2,Омск — Тюмень,620.0,2026-03-03,2026-03-04,2,Петров Алексей Сергеевич,Средний стаж,10.0,14500.0,1500.0,17450.0
2,2,Омск — Тюмень,620.0,2026-03-03,2026-03-04,3,Сидоров Дмитрий Андреевич,Опытный водитель,25.0,14500.0,1000.0,19125.0
3,3,Омск — Екатеринбург,950.0,2026-03-05,2026-03-07,3,Сидоров Дмитрий Андреевич,Опытный водитель,25.0,22000.0,2000.0,29500.0
4,4,Новосибирск — Красноярск,790.0,2026-03-08,2026-03-10,4,Кузнецов Олег Викторович,Высокий стаж,40.0,18500.0,2500.0,28400.0
5,4,Новосибирск — Красноярск,790.0,2026-03-08,2026-03-10,5,Смирнов Николай Игоревич,Высокий стаж,40.0,18500.0,2000.0,27900.0
6,5,Омск — Москва,2700.0,2026-03-11,2026-03-15,5,Смирнов Николай Игоревич,Высокий стаж,40.0,55000.0,5000.0,82000.0


покажет общую сумму выплат каждому водителю уже через представление

In [28]:
show_query("""
SELECT
    driver_id,
    driver_full_name,
    experience_level,
    percentage_rate,
    COUNT(shipment_id) AS shipments_count,
    SUM(final_payment) AS total_payment
FROM shipment_payment_view
GROUP BY
    driver_id,
    driver_full_name,
    experience_level,
    percentage_rate
ORDER BY total_payment DESC;
""")

,driver_id,driver_full_name,experience_level,percentage_rate,shipments_count,total_payment
0,5,Смирнов Николай Игоревич,Высокий стаж,40.0,2,109900.0
1,3,Сидоров Дмитрий Андреевич,Опытный водитель,25.0,2,48625.0
2,4,Кузнецов Олег Викторович,Высокий стаж,40.0,1,28400.0
3,2,Петров Алексей Сергеевич,Средний стаж,10.0,1,17450.0
4,1,Иванов Иван Петрович,Начальный стаж,0.0,1,16000.0


## Создание хранимой функции для расчета выплаты водителю

Хранимая функция используется для расчета итоговой выплаты водителю. 
Расчет выполняется на стороне PostgreSQL с учетом базовой оплаты маршрута, стажа водителя и премии.

In [29]:
try:
    with engine.begin() as connection:
        connection.execute(text("""
            DROP FUNCTION IF EXISTS calculate_driver_payment(NUMERIC, INTEGER, NUMERIC);
        """))

        connection.execute(text("""
            CREATE OR REPLACE FUNCTION calculate_driver_payment(
                p_base_salary NUMERIC,
                p_experience_level_id INTEGER,
                p_bonus NUMERIC
            )
            RETURNS NUMERIC
            LANGUAGE plpgsql
            AS $$
            DECLARE
                v_percentage_rate NUMERIC;
                v_final_payment NUMERIC;
            BEGIN
                SELECT percentage_rate
                INTO v_percentage_rate
                FROM experience_coefficients
                WHERE experience_level_id = p_experience_level_id;

                IF v_percentage_rate IS NULL THEN
                    RAISE EXCEPTION 'Не найден уровень стажа с ID: %', p_experience_level_id;
                END IF;

                v_final_payment := ROUND(
                    p_base_salary + p_base_salary * v_percentage_rate / 100 + COALESCE(p_bonus, 0),
                    2
                );

                RETURN v_final_payment;
            END;
            $$;
        """))

    print("Функция calculate_driver_payment успешно создана")

except SQLAlchemyError as error:
    print("Ошибка при создании функции:")
    print(error)

Функция calculate_driver_payment успешно создана


Проверяем функцию

In [30]:
show_query("""
SELECT calculate_driver_payment(15000, 2, 1000) AS payment_result;
""")

,payment_result
0,17500.0


Функция `calculate_driver_payment` позволяет централизованно выполнять расчет выплаты водителю. 
Это удобно, так как при изменении правил расчета достаточно изменить функцию, а не переписывать все SQL-запросы.

## Создание хранимой процедуры для начисления премии водителю

Хранимая процедура используется для изменения размера премии водителя за конкретную перевозку. 
Процедура проверяет существование записи о перевозке и водителе, после чего увеличивает премию на указанную сумму.

In [31]:
try:
    with engine.begin() as connection:
        connection.execute(text("""
            DROP PROCEDURE IF EXISTS add_bonus_to_driver(INTEGER, INTEGER, NUMERIC);
        """))

        connection.execute(text("""
            CREATE OR REPLACE PROCEDURE add_bonus_to_driver(
                p_shipment_id INTEGER,
                p_driver_id INTEGER,
                p_bonus_amount NUMERIC
            )
            LANGUAGE plpgsql
            AS $$
            BEGIN
                IF p_bonus_amount < 0 THEN
                    RAISE EXCEPTION 'Размер премии не может быть отрицательным';
                END IF;

                UPDATE shipment_drivers
                SET bonus = bonus + p_bonus_amount
                WHERE shipment_id = p_shipment_id
                  AND driver_id = p_driver_id;

                IF NOT FOUND THEN
                    RAISE EXCEPTION 
                    'Связь перевозки % и водителя % не найдена', 
                    p_shipment_id, p_driver_id;
                END IF;
            END;
            $$;
        """))

    print("Процедура add_bonus_to_driver успешно создана")

except SQLAlchemyError as error:
    print("Ошибка при создании процедуры:")
    print(error)

Процедура add_bonus_to_driver успешно создана


премии до изменения

In [32]:
show_query("""
SELECT
    sd.shipment_id,
    sd.driver_id,
    d.full_name AS driver_full_name,
    sd.bonus
FROM shipment_drivers sd
JOIN drivers d ON sd.driver_id = d.driver_id
ORDER BY sd.shipment_id, sd.driver_id;
""")

,shipment_id,driver_id,driver_full_name,bonus
0,1,1,Иванов Иван Петрович,1000.0
1,2,2,Петров Алексей Сергеевич,1500.0
2,2,3,Сидоров Дмитрий Андреевич,1000.0
3,3,3,Сидоров Дмитрий Андреевич,2000.0
4,4,4,Кузнецов Олег Викторович,2500.0
5,4,5,Смирнов Николай Игоревич,2000.0
6,5,5,Смирнов Николай Игоревич,5000.0


In [33]:
try:
    with engine.begin() as connection:
        connection.execute(text("""
            CALL add_bonus_to_driver(1, 1, 500);
        """))

    print("Премия водителю успешно увеличена")

except SQLAlchemyError as error:
    print("Ошибка при вызове процедуры:")
    print(error)

Премия водителю успешно увеличена


после процедуры

In [34]:
show_query("""
SELECT
    sd.shipment_id,
    sd.driver_id,
    d.full_name AS driver_full_name,
    sd.bonus
FROM shipment_drivers sd
JOIN drivers d ON sd.driver_id = d.driver_id
WHERE sd.shipment_id = 1
  AND sd.driver_id = 1;
""")

,shipment_id,driver_id,driver_full_name,bonus
0,1,1,Иванов Иван Петрович,1500.0


Проверка, что итоговая выплата тоже изменилась

In [35]:
show_query("""
SELECT
    shipment_id,
    route_name,
    driver_full_name,
    base_salary,
    percentage_rate,
    bonus,
    final_payment
FROM shipment_payment_view
WHERE shipment_id = 1
  AND driver_id = 1;
""")

,shipment_id,route_name,driver_full_name,base_salary,percentage_rate,bonus,final_payment
0,1,Омск — Новосибирск,Иванов Иван Петрович,15000.0,0.0,1500.0,16500.0


## Создание триггера для контроля количества водителей в перевозке

По условию предметной области одну перевозку могут выполнять один или два водителя. 
Для контроля этого ограничения создается триггер, который запрещает добавлять более двух водителей к одной перевозке.

In [36]:
try:
    with engine.begin() as connection:
        connection.execute(text("""
            DROP TRIGGER IF EXISTS check_max_two_drivers_trigger ON shipment_drivers;
        """))

        connection.execute(text("""
            DROP FUNCTION IF EXISTS check_max_two_drivers();
        """))

        connection.execute(text("""
            CREATE OR REPLACE FUNCTION check_max_two_drivers()
            RETURNS TRIGGER
            LANGUAGE plpgsql
            AS $$
            DECLARE
                drivers_count INTEGER;
            BEGIN
                SELECT COUNT(*)
                INTO drivers_count
                FROM shipment_drivers
                WHERE shipment_id = NEW.shipment_id;

                IF drivers_count >= 2 THEN
                    RAISE EXCEPTION 
                    'Нельзя добавить больше двух водителей к одной перевозке. ID перевозки: %', 
                    NEW.shipment_id;
                END IF;

                RETURN NEW;
            END;
            $$;
        """))

    print("Триггерная функция check_max_two_drivers успешно создана")

except SQLAlchemyError as error:
    print("Ошибка при создании триггерной функции:")
    print(error)

Триггерная функция check_max_two_drivers успешно создана


In [37]:
try:
    with engine.begin() as connection:
        connection.execute(text("""
            CREATE TRIGGER check_max_two_drivers_trigger
            BEFORE INSERT ON shipment_drivers
            FOR EACH ROW
            EXECUTE FUNCTION check_max_two_drivers();
        """))

    print("Триггер check_max_two_drivers_trigger успешно создан")

except SQLAlchemyError as error:
    print("Ошибка при создании триггера:")
    print(error)

Триггер check_max_two_drivers_trigger успешно создан


Проверяем, сколько водителей сейчас у перевозок

In [38]:
show_query("""
SELECT
    s.shipment_id,
    r.name AS route_name,
    COUNT(sd.driver_id) AS drivers_count
FROM shipments s
JOIN routes r ON s.route_id = r.route_id
LEFT JOIN shipment_drivers sd ON s.shipment_id = sd.shipment_id
GROUP BY s.shipment_id, r.name
ORDER BY s.shipment_id;
""")

,shipment_id,route_name,drivers_count
0,1,Омск — Новосибирск,1
1,2,Омск — Тюмень,2
2,3,Омск — Екатеринбург,1
3,4,Новосибирск — Красноярск,2
4,5,Омск — Москва,1


к перевозке id = 1 пробуем добавить сразу двух дополнительных водителей. Второй INSERT должен упасть, потому что водителей стало бы больше двух

In [39]:
try:
    with engine.begin() as connection:
        connection.execute(text("""
            INSERT INTO shipment_drivers (shipment_id, driver_id, bonus)
            VALUES (1, 2, 0);
        """))

        connection.execute(text("""
            INSERT INTO shipment_drivers (shipment_id, driver_id, bonus)
            VALUES (1, 3, 0);
        """))

    print("Тестовые водители добавлены")

except SQLAlchemyError as error:
    print("Триггер сработал корректно:")
    print(error)

Триггер сработал корректно:
(psycopg2.errors.RaiseException) ОШИБКА:  Нельзя добавить больше двух водителей к одной перевозке. ID перевозки: 1
CONTEXT:  функция PL/pgSQL check_max_two_drivers(), строка 11, оператор RAISE

[SQL: 
            INSERT INTO shipment_drivers (shipment_id, driver_id, bonus)
            VALUES (1, 3, 0);
        ]
(Background on this error at: https://sqlalche.me/e/20/2j85)


Проверяем, что лишние данные не добавились

In [40]:
show_query("""
SELECT
    sd.shipment_id,
    sd.driver_id,
    d.full_name AS driver_full_name,
    sd.bonus
FROM shipment_drivers sd
JOIN drivers d ON sd.driver_id = d.driver_id
WHERE sd.shipment_id = 1
ORDER BY sd.driver_id;
""")

,shipment_id,driver_id,driver_full_name,bonus
0,1,1,Иванов Иван Петрович,1500.0


## Обработка исключений

В данном разделе демонстрируется перехват исключений, которые могут возникнуть при работе с базой данных. 
Обработка ошибок позволяет не завершать программу аварийно, а выводить понятное сообщение пользователю.

Пример 1 — попытка добавить маршрут с отрицательным расстоянием

In [41]:
try:
    session = SessionLocal()

    wrong_route = Route(
        name="Ошибочный маршрут",
        distance=-100,
        base_salary=10000
    )

    session.add(wrong_route)
    session.commit()

    print("Маршрут добавлен")

except SQLAlchemyError as error:
    session.rollback()
    print("Ошибка перехвачена корректно:")
    print(error)

finally:
    session.close()

Ошибка перехвачена корректно:
(psycopg2.errors.CheckViolation) ОШИБКА:  новая строка в отношении "routes" нарушает ограничение-проверку "check_route_distance_positive"
DETAIL:  Ошибочная строка содержит (6, Ошибочный маршрут, -100.00, 10000.00).

[SQL: INSERT INTO routes (name, distance, base_salary) VALUES (%(name)s, %(distance)s, %(base_salary)s) RETURNING routes.route_id]
[parameters: {'name': 'Ошибочный маршрут', 'distance': -100, 'base_salary': 10000}]
(Background on this error at: https://sqlalche.me/e/20/gkpj)


Пример 2 — попытка добавить водителя с отрицательным стажем

In [42]:
try:
    session = SessionLocal()

    wrong_driver = Driver(
        full_name="Ошибкин Петр Иванович",
        experience_level_id=999
    )

    session.add(wrong_driver)
    session.commit()

    print("Водитель добавлен")

except SQLAlchemyError as error:
    session.rollback()
    print("Ошибка перехвачена корректно:")
    print(error)

finally:
    session.close()

Ошибка перехвачена корректно:
(psycopg2.errors.ForeignKeyViolation) ОШИБКА:  INSERT или UPDATE в таблице "drivers" нарушает ограничение внешнего ключа "drivers_experience_level_id_fkey"
DETAIL:  Ключ (experience_level_id)=(999) отсутствует в таблице "experience_coefficients".

[SQL: INSERT INTO drivers (experience_level_id, full_name) VALUES (%(experience_level_id)s, %(full_name)s) RETURNING drivers.driver_id]
[parameters: {'experience_level_id': 999, 'full_name': 'Ошибкин Петр Иванович'}]
(Background on this error at: https://sqlalche.me/e/20/gkpj)


Пример 3 — попытка вызвать процедуру с отрицательной премией

In [43]:
try:
    with engine.begin() as connection:
        connection.execute(text("""
            CALL add_bonus_to_driver(1, 1, -1000);
        """))

    print("Премия начислена")

except SQLAlchemyError as error:
    print("Ошибка перехвачена корректно:")
    print(error)

Ошибка перехвачена корректно:
(psycopg2.errors.RaiseException) ОШИБКА:  Размер премии не может быть отрицательным
CONTEXT:  функция PL/pgSQL add_bonus_to_driver(integer,integer,numeric), строка 4, оператор RAISE

[SQL: 
            CALL add_bonus_to_driver(1, 1, -1000);
        ]
(Background on this error at: https://sqlalche.me/e/20/2j85)


## Итоговая проверка данных

На данном этапе выполняется итоговая проверка содержимого основных таблиц базы данных.

In [44]:
tables = {
    "routes": "route_id",
    "experience_coefficients": "experience_level_id",
    "drivers": "driver_id",
    "shipments": "shipment_id",
    "shipment_drivers": "shipment_id, driver_id",
}

for table, order_columns in tables.items():
    print(f"\nТаблица: {table}")
    display(
        show_query(
            f"SELECT * FROM {table} ORDER BY {order_columns};"
        )
    )


Таблица: routes


,route_id,name,distance,base_salary
0,1,Омск — Новосибирск,650.0,15000.0
1,2,Омск — Тюмень,620.0,14500.0
2,3,Омск — Екатеринбург,950.0,22000.0
3,4,Новосибирск — Красноярск,790.0,18500.0
4,5,Омск — Москва,2700.0,55000.0



Таблица: experience_coefficients


,experience_level_id,percentage_rate,description
0,1,0.0,Начальный стаж
1,2,10.0,Средний стаж
2,3,25.0,Опытный водитель
3,4,40.0,Высокий стаж



Таблица: drivers


,driver_id,experience_level_id,full_name
0,1,1,Иванов Иван Петрович
1,2,2,Петров Алексей Сергеевич
2,3,3,Сидоров Дмитрий Андреевич
3,4,4,Кузнецов Олег Викторович
4,5,4,Смирнов Николай Игоревич



Таблица: shipments


,shipment_id,route_id,departure_date,arrival_date
0,1,1,2026-03-01,2026-03-02
1,2,2,2026-03-03,2026-03-04
2,3,3,2026-03-05,2026-03-07
3,4,4,2026-03-08,2026-03-10
4,5,5,2026-03-11,2026-03-15



Таблица: shipment_drivers


,shipment_id,driver_id,bonus
0,1,1,1500.0
1,2,2,1500.0
2,2,3,1000.0
3,3,3,2000.0
4,4,4,2500.0
5,4,5,2000.0
6,5,5,5000.0


## Итоговый расчет выплат водителям

Выполним итоговый запрос, который показывает выплаты водителям по всем перевозкам с учетом маршрута, стажа и премии.

In [45]:
show_query("""
SELECT
    shipment_id,
    route_name,
    driver_full_name,
    experience_level,
    percentage_rate,
    base_salary,
    bonus,
    final_payment
FROM shipment_payment_view
ORDER BY shipment_id, driver_full_name;
""")

,shipment_id,route_name,driver_full_name,experience_level,percentage_rate,base_salary,bonus,final_payment
0,1,Омск — Новосибирск,Иванов Иван Петрович,Начальный стаж,0.0,15000.0,1500.0,16500.0
1,2,Омск — Тюмень,Петров Алексей Сергеевич,Средний стаж,10.0,14500.0,1500.0,17450.0
2,2,Омск — Тюмень,Сидоров Дмитрий Андреевич,Опытный водитель,25.0,14500.0,1000.0,19125.0
3,3,Омск — Екатеринбург,Сидоров Дмитрий Андреевич,Опытный водитель,25.0,22000.0,2000.0,29500.0
4,4,Новосибирск — Красноярск,Кузнецов Олег Викторович,Высокий стаж,40.0,18500.0,2500.0,28400.0
5,4,Новосибирск — Красноярск,Смирнов Николай Игоревич,Высокий стаж,40.0,18500.0,2000.0,27900.0
6,5,Омск — Москва,Смирнов Николай Игоревич,Высокий стаж,40.0,55000.0,5000.0,82000.0


Общая сумма выплат фирмы

In [46]:
show_query("""
SELECT
    SUM(final_payment) AS total_company_driver_payments
FROM shipment_payment_view;
""")

,total_company_driver_payments
0,220875.0


Рейтинг водителей по заработку

In [47]:
show_query("""
SELECT
    driver_full_name,
    experience_level,
    percentage_rate,
    COUNT(shipment_id) AS shipments_count,
    SUM(final_payment) AS total_payment
FROM shipment_payment_view
GROUP BY
    driver_full_name,
    experience_level,
    percentage_rate
ORDER BY total_payment DESC;
""")

,driver_full_name,experience_level,percentage_rate,shipments_count,total_payment
0,Смирнов Николай Игоревич,Высокий стаж,40.0,2,109900.0
1,Сидоров Дмитрий Андреевич,Опытный водитель,25.0,2,48625.0
2,Кузнецов Олег Викторович,Высокий стаж,40.0,1,28400.0
3,Петров Алексей Сергеевич,Средний стаж,10.0,1,17450.0
4,Иванов Иван Петрович,Начальный стаж,0.0,1,16500.0


## Вывод

В ходе выполнения расчетно-графической работы была спроектирована и реализована реляционная база данных для предметной области «Грузовые перевозки».

В базе данных были созданы таблицы для хранения информации о маршрутах, водителях, коэффициентах оплаты по стажу, перевозках и участии водителей в перевозках.

С помощью SQLAlchemy ORM были реализованы модели данных, выполнено создание таблиц в PostgreSQL, добавление тестовых данных и CRUD-операции.

Также были реализованы сложные запросы на выборку данных, представление для расчета выплат водителям, хранимая функция расчета оплаты, процедура начисления премии и триггер, запрещающий добавлять более двух водителей к одной перевозке.

В работе также была показана обработка исключений при нарушении ограничений базы данных.

Таким образом, разработанная база данных позволяет хранить сведения о грузовых перевозках и выполнять расчет выплат водителям с учетом маршрута, стажа и премий.